# 03 — Beta Diversity (SGA vs AGA and FGR vs AGA)

**Manuscript:** Associations between maternal gut microbiome diversity in early and late pregnancy and small-for-gestational-age or growth restricted births: a nested case-control study

**Purpose:** Assesses maternal gut microbiota community composition (β-diversity) at TP1 and TP2 in two comparisons (SGA vs AGA and FGR vs AGA), using Bray–Curtis, Jaccard and Aitchison (robust CLR) distances. Tests homogeneity of group dispersions (betadisper), runs crude and adjusted PERMANOVA (adonis2), and visualizes community composition in PCoA plots.

**Produces:** Figure S2 and Table S2

**Programmer:** Anusha Antony

# Load data

In [ ]:
library(dplyr)
library(tidyverse)
library(vegan)
library(ggplot2)
library(patchwork)
library(stringr)
library(patchwork)
library(cowplot)

In [ ]:
# Define base path 
base_path_1 <- "data"
base_path_2 <- "data"
base_path_3 <- "data"

# Functions to call

In [ ]:
# Functions used

# Classified taxa rarely sum to 100% per sample, since low-abundance taxa are dropped during filtering. 
#This adds an "Other_taxa" row with the missing fraction so every sample sums to 100% for plotting and comparison.
add_other_taxa <- function(taxa_table, min_total = 100, row_name = "Other_taxa") {
  
  col_totals <- colSums(taxa_table)
  new_row <- ifelse(col_totals < min_total,
                    min_total - col_totals,
                    0)

  new_row <- setNames(new_row, names(taxa_table))
  
  # Convert to one-row data frame
  other_taxa_row <- as.data.frame(t(new_row))
  rownames(other_taxa_row) <- row_name
  
  # Add row to original table
  new_table <- rbind(taxa_table, other_taxa_row)
  
  # Return result
  return(as.data.frame(new_table))
}


In [ ]:
# Function for each distance metric
find_jaccard <- function(taxa) {
  vegdist(decostand(as.matrix(taxa), method = "pa"), method = "jaccard")
}

find_aitchison <- function(taxa) {
  vegdist(as.matrix(taxa), method = "robust.aitchison")
}

find_bray <- function(taxa) {
  vegdist(as.matrix(taxa), method = "bray")
}

In [ ]:
# Function to check the dispersion
check_dispersion <- function(dist, group, title = "") {
  disp <- betadisper(dist, group)
  test <- permutest(disp, permutations = 999)
  print(test)
  boxplot(disp, main = title)
  plot(disp, main = title)
  invisible(test)
}

In [ ]:
# Function to run the adonis2 test
run_crude_adonis <- function(dist, meta, seed = 1234) {
  set.seed(seed)
  adonis2(dist ~ Group, data = meta, permutations = 999)
}

run_adjusted_adonis <- function(dist, meta, confounder, seed = 1234) {
  set.seed(seed)
  f <- reformulate(c("Group", confounder), response = "dist")
  adonis2(f, data = meta, permutations = 999, by = "margin", na.action = na.omit)
}

# SGA Vs AGA

In [ ]:
sga_tp1 <- read.csv(file.path(base_path_1,"data"))
sga_tp2 <- read.csv(file.path(base_path_1, "data"))

In [ ]:
# ensure factors exist and set reference explicitly
sga_tp1$Age_above_35 <- factor(sga_tp1$Age_above_35, levels = c("0","1"))

sga_tp1$ATA_BMI_group_prior <- factor(sga_tp1$ATA_BMI_group_prior,
                                          levels = c("Normalweight","Underweight","Overweight","Obese"))
sga_tp1$Primipara <- factor(sga_tp1$Primipara, levels = c("0","1"))

sga_tp1$Q1_X62_Alcohol_3_months_before_pregnancy <- factor(sga_tp1$Q1_X62_Alcohol_3_months_before_pregnancy,
                                                               levels = c("0","1"))
sga_tp1$Q1_X31_Diagnosed_PCOS <- factor(sga_tp1$Q1_X31_Diagnosed_PCOS, levels = c("0","1"))

sga_tp1$Insufficient_weight_gain <- factor(sga_tp1$Insufficient_weight_gain, levels = c("No","Yes"))

sga_tp1$Q2_X15_during_bloodpressure_medication <- factor(sga_tp1$Q2_X15_during_bloodpressure_medication,levels = c("0","1"))  
sga_tp1$Q1_X77_during_bloodpressure_medication <- factor(sga_tp1$Q1_X77_during_bloodpressure_medication,levels = c("0","1"))                                                           

sga_tp1$preeclampsia_health_reg_ATA <- factor(sga_tp1$preeclampsia_health_reg_ATA, levels = c("No","Yes"))
sga_tp1$Group <- factor(sga_tp1$Group ,levels = c("Control","Case-SGA"))


In [ ]:
sga_tp1$kit1.faecal_sample.barcode <- as.character(sga_tp1$kit1.faecal_sample.barcode )
sga_tp1$kit2.faecal_sample.barcode <- as.character(sga_tp1$kit2.faecal_sample.barcode )

In [ ]:
sga_tp2_final <- sga_tp1 %>% 
                        filter(kit2.faecal_sample.barcode %in% sga_tp2$kit2.faecal_sample.barcode)
head(sga_tp2_final)
dim(sga_tp2_final)

In [ ]:
SGA_tp1_taxa <- read.table(file.path(base_path_2, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)
SGA_tp2_taxa <- read.table(file.path(base_path_2, "data") ,sep= "\t",
  header      = TRUE,
  check.names = FALSE)

In [ ]:
SGA_tp1_taxa_full <- add_other_taxa(SGA_tp1_taxa)

SGA_tp2_taxa_full <- add_other_taxa(SGA_tp2_taxa)


In [ ]:
tail(SGA_tp1_taxa_full)
colSums(SGA_tp1_taxa_full)
colSums(SGA_tp2_taxa_full)

In [ ]:
SGA_tp1_taxa_full <- as.data.frame(t(SGA_tp1_taxa_full))
SGA_tp2_taxa_full <- as.data.frame(t(SGA_tp2_taxa_full))

In [ ]:
head(SGA_tp1_taxa_full )

## 1. Compute distance matrices (vegdist)

### TP1

In [ ]:

SGA_tp1_taxa_full_update <- SGA_tp1_taxa_full[sga_tp1$kit1.faecal_sample.barcode, , drop = FALSE]
head(SGA_tp1_taxa_full_update)

In [ ]:
head(sga_tp1)

In [ ]:
identical(rownames(SGA_tp1_taxa_full_update ), sga_tp1$kit1.faecal_sample.barcode)

In [ ]:
j_SGA_tp1  <- find_jaccard(SGA_tp1_taxa_full_update)
a_SGA_tp1  <- find_aitchison(SGA_tp1_taxa_full_update)
bc_SGA_tp1 <- find_bray(SGA_tp1_taxa_full_update)

### TP2

In [ ]:
SGA_tp2_taxa_full_update <-SGA_tp2_taxa_full[sga_tp2_final$kit2.faecal_sample.barcode, , drop = FALSE]
head(SGA_tp2_taxa_full_update)

In [ ]:
head(sga_tp2_final)

In [ ]:
identical(rownames(SGA_tp2_taxa_full_update ), sga_tp2_final$kit2.faecal_sample.barcode)

In [ ]:
#Call the function
j_SGA_tp2  <- find_jaccard(SGA_tp2_taxa_full_update)
a_SGA_tp2  <- find_aitchison(SGA_tp2_taxa_full_update)
bc_SGA_tp2 <- find_bray(SGA_tp2_taxa_full_update)

## 2. betadisper dispersion test

### TP1

In [ ]:
set.seed(123)
check_dispersion(j_SGA_tp1, sga_tp1$Group, "Jaccard – SGA vs AGA, TP1")

In [ ]:
set.seed(123)
check_dispersion(a_SGA_tp1, sga_tp1$Group, "Aitchison – SGA vs AGA, TP1")

In [ ]:
set.seed(123)
check_dispersion(bc_SGA_tp1, sga_tp1$Group, "BrayCurtis – SGA vs AGA, TP1")

### TP2

In [ ]:
set.seed(123)
check_dispersion(j_SGA_tp2, sga_tp2_final$Group, "Jaccard – SGA vs AGA, TP2")

In [ ]:
set.seed(123)
check_dispersion(a_SGA_tp2, sga_tp2_final$Group, "Aitchison – SGA vs AGA, TP2")

In [ ]:
set.seed(123)
check_dispersion(bc_SGA_tp2, sga_tp2_final$Group, "BrayCurtis – SGA vs AGA, TP2")

## 3. PERMANOVA (unadjusted and adjusted) 

### TP1

In [ ]:
confounder_tp1 <- c("Age_above_35",
             "ATA_BMI_group_prior",
             "Primipara",
             "Q1_X62_Alcohol_3_months_before_pregnancy",
             "Q1_X31_Diagnosed_PCOS",
             "Insufficient_weight_gain",
             "Q1_X77_during_bloodpressure_medication")


In [ ]:
# Crude PERMANOVA
j_crude_sga_tp1 <- run_crude_adonis(j_SGA_tp1,  sga_tp1)
j_crude_sga_tp1

a_crude_sga_tp1 <- run_crude_adonis(a_SGA_tp1,  sga_tp1)
a_crude_sga_tp1

bc_crude_sga_tp1 <- run_crude_adonis(bc_SGA_tp1,  sga_tp1)
bc_crude_sga_tp1


In [ ]:
# Adjusted PERMANOVA
j_adjust_sga_tp1 <- run_adjusted_adonis(j_SGA_tp1, sga_tp1, confounder_tp1)
a_adjust_sga_tp1 <- run_adjusted_adonis(a_SGA_tp1, sga_tp1, confounder_tp1)
bc_adjust_sga_tp1 <- run_adjusted_adonis(bc_SGA_tp1, sga_tp1, confounder_tp1)

In [ ]:
j_adjust_sga_tp1 
a_adjust_sga_tp1
bc_adjust_sga_tp1

### TP2

In [ ]:
confounder_tp2 <- c("Age_above_35",
             "ATA_BMI_group_prior",
             "Primipara",
             "Q1_X62_Alcohol_3_months_before_pregnancy",
             "Q1_X31_Diagnosed_PCOS",
             "Insufficient_weight_gain",
             "Q2_X15_during_bloodpressure_medication", "preeclampsia_health_reg_ATA")


In [ ]:
# Crude PERMANOVA
j_crude_sga_tp2 <- run_crude_adonis(j_SGA_tp2,  sga_tp2_final)
j_crude_sga_tp2

a_crude_sga_tp2 <- run_crude_adonis(a_SGA_tp2,  sga_tp2_final)
a_crude_sga_tp2

bc_crude_sga_tp2 <- run_crude_adonis(bc_SGA_tp2,  sga_tp2_final)
bc_crude_sga_tp2

In [ ]:
# Adjusted PERMANOVA
j_adjust_sga_tp2 <- run_adjusted_adonis(j_SGA_tp2, sga_tp2_final, confounder_tp2)
a_adjust_sga_tp2 <- run_adjusted_adonis(a_SGA_tp2, sga_tp2_final, confounder_tp2)
bc_adjust_sga_tp2 <- run_adjusted_adonis(bc_SGA_tp2, sga_tp2_final, confounder_tp2)

In [ ]:
j_adjust_sga_tp2
a_adjust_sga_tp2 
bc_adjust_sga_tp2 

# FGR vs AGA

In [ ]:
fgr_tp1 <- read.csv(file.path(base_path_1,"data"))
fgr_tp2 <- read.csv(file.path(base_path_1, "data"))

In [ ]:
# ensure factors exist and set reference explicitly
fgr_tp1$Age_above_35 <- factor(fgr_tp1$Age_above_35, levels = c("0","1"))

fgr_tp1$ATA_BMI_group_prior <- factor(fgr_tp1$ATA_BMI_group_prior,
                                          levels = c("Normalweight","Underweight","Overweight","Obese"))
fgr_tp1$Primipara <- factor(fgr_tp1$Primipara, levels = c("0","1"))

fgr_tp1$Q1_X62_Alcohol_3_months_before_pregnancy <- factor(fgr_tp1$Q1_X62_Alcohol_3_months_before_pregnancy,
                                                               levels = c("0","1"))
fgr_tp1$Q1_X31_Diagnosed_PCOS <- factor(fgr_tp1$Q1_X31_Diagnosed_PCOS, levels = c("0","1"))

fgr_tp1$Insufficient_weight_gain <- factor(fgr_tp1$Insufficient_weight_gain, levels = c("No","Yes"))

fgr_tp1$Q2_X15_during_bloodpressure_medication <- factor(fgr_tp1$Q2_X15_during_bloodpressure_medication,levels = c("0","1"))  
fgr_tp1$Q1_X77_during_bloodpressure_medication <- factor(fgr_tp1$Q1_X77_during_bloodpressure_medication,levels = c("0","1"))                                                           

fgr_tp1$preeclampsia_health_reg_ATA <- factor(fgr_tp1$preeclampsia_health_reg_ATA, levels = c("No","Yes"))
fgr_tp1$Group <- factor(fgr_tp1$Group ,levels = c("Control","Case-FGR"))


In [ ]:
fgr_tp1$kit1.faecal_sample.barcode <- as.character(fgr_tp1$kit1.faecal_sample.barcode )
fgr_tp1$kit2.faecal_sample.barcode <- as.character(fgr_tp1$kit2.faecal_sample.barcode )

In [ ]:
fgr_tp2_final <- fgr_tp1 %>% 
                        filter(kit2.faecal_sample.barcode %in% fgr_tp2$kit2.faecal_sample.barcode)
head(fgr_tp2_final)
dim(fgr_tp2_final)

In [ ]:
FGR_tp1_taxa <- read.table(file.path(base_path_3, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)

FGR_tp2_taxa <- read.table(file.path(base_path_3, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)

In [ ]:
FGR_tp1_taxa_full <- add_other_taxa(FGR_tp1_taxa)
FGR_tp2_taxa_full <- add_other_taxa(FGR_tp2_taxa)

In [ ]:
#colSums(FGR_tp2_taxa_full)

In [ ]:
FGR_tp1_taxa_full <- as.data.frame(t(FGR_tp1_taxa_full))
FGR_tp2_taxa_full  <- as.data.frame(t(FGR_tp2_taxa_full))

In [ ]:
head(FGR_tp1_taxa_full)

## 1. Compute distance matrices (vegdist)

### TP1

In [ ]:
FGR_tp1_taxa_full_update <- FGR_tp1_taxa_full[fgr_tp1$kit1.faecal_sample.barcode, , drop = FALSE]
head(FGR_tp1_taxa_full_update)

In [ ]:
identical(rownames(FGR_tp1_taxa_full_update ), fgr_tp1$kit1.faecal_sample.barcode)

In [ ]:
j_FGR_tp1  <- find_jaccard(FGR_tp1_taxa_full_update)
a_FGR_tp1  <- find_aitchison(FGR_tp1_taxa_full_update)
bc_FGR_tp1 <- find_bray(FGR_tp1_taxa_full_update)

### TP2

In [ ]:
FGR_tp2_taxa_full_update <-FGR_tp2_taxa_full[fgr_tp2_final$kit2.faecal_sample.barcode, , drop = FALSE]
head(FGR_tp2_taxa_full_update)

In [ ]:
identical(rownames(FGR_tp2_taxa_full_update ), fgr_tp2_final$kit2.faecal_sample.barcode)

In [ ]:
j_FGR_tp2  <- find_jaccard(FGR_tp2_taxa_full_update)
a_FGR_tp2  <- find_aitchison(FGR_tp2_taxa_full_update)
bc_FGR_tp2 <- find_bray(FGR_tp2_taxa_full_update)

## 2. betadisper dispersion test

### TP1

In [ ]:
set.seed(123)
check_dispersion(j_FGR_tp1, fgr_tp1$Group, "Jaccard – FGR vs AGA, TP1")

In [ ]:
set.seed(123)
check_dispersion(a_FGR_tp1, fgr_tp1$Group, "Aitchison – FGR vs AGA, TP1")

In [ ]:
set.seed(123)
check_dispersion(bc_FGR_tp1, fgr_tp1$Group, "BrayCurtis – FGR vs AGA, TP1")

### TP2

In [ ]:
set.seed(123)
check_dispersion(j_FGR_tp2, fgr_tp2_final$Group, "Jaccard – FGR vs AGA, TP2")

In [ ]:
set.seed(123)
check_dispersion(a_FGR_tp2, fgr_tp2_final$Group, " Aitchison – FGR vs AGA, TP2")

In [ ]:
set.seed(123)
check_dispersion(bc_FGR_tp2, fgr_tp2_final$Group, " BrayCurtis– FGR vs AGA, TP2")

## 3. PERMANOVA (unadjusted and adjusted) 

### TP1

In [ ]:
# Crude PERMANOVA
j_crude_fgr_tp1 <- run_crude_adonis(j_FGR_tp1,  fgr_tp1)
j_crude_fgr_tp1

a_crude_fgr_tp1 <- run_crude_adonis(a_FGR_tp1,  fgr_tp1)
a_crude_fgr_tp1

bc_crude_fgr_tp1 <- run_crude_adonis(bc_FGR_tp1,  fgr_tp1)
bc_crude_fgr_tp1

In [ ]:
# Adjusted PERMANOVA
j_adjust_fgr_tp1 <- run_adjusted_adonis(j_FGR_tp1, fgr_tp1, confounder_tp1)
a_adjust_fgr_tp1 <- run_adjusted_adonis(a_FGR_tp1, fgr_tp1, confounder_tp1)
bc_adjust_fgr_tp1 <- run_adjusted_adonis(bc_FGR_tp1, fgr_tp1, confounder_tp1)

In [ ]:
j_adjust_fgr_tp1 
a_adjust_fgr_tp1
bc_adjust_fgr_tp1

### TP2

In [ ]:
# Crude PERMANOVA
j_crude_fgr_tp2 <- run_crude_adonis(j_FGR_tp2,  fgr_tp2_final)
j_crude_fgr_tp2

a_crude_fgr_tp2 <- run_crude_adonis(a_FGR_tp2,  fgr_tp2_final)
a_crude_fgr_tp2

bc_crude_fgr_tp2 <- run_crude_adonis(bc_FGR_tp2,  fgr_tp2_final)
bc_crude_fgr_tp2

In [ ]:
# Adjusted PERMANOVA
j_adjust_fgr_tp2 <- run_adjusted_adonis(j_FGR_tp2, fgr_tp2_final, confounder_tp2)
a_adjust_fgr_tp2 <- run_adjusted_adonis(a_FGR_tp2, fgr_tp2_final, confounder_tp2)
bc_adjust_fgr_tp2<- run_adjusted_adonis(bc_FGR_tp2, fgr_tp2_final, confounder_tp2)

In [ ]:
j_adjust_fgr_tp2
a_adjust_fgr_tp2
bc_adjust_fgr_tp2

# Figure S2

In [ ]:
bc_adjust_fgr_tp2["Group", c("R2", "Pr(>F)")]

In [ ]:
#Extract the R2 and P value from the model
#bc_adjust_fgr_tp2["Group", c("R2", "Pr(>F)")]
get_values <- function(adonis_model) {
    c(R2 = adonis_model["Group", "R2"], p = adonis_model["Group", "Pr(>F)"])
}

In [ ]:

bc_p_sga_tp1 <- get_values(bc_adjust_sga_tp1)
bc_p_sga_tp2 <- get_values(bc_adjust_sga_tp2)
bc_p_fgr_tp1 <- get_values(bc_adjust_fgr_tp1)
bc_p_fgr_tp2 <- get_values(bc_adjust_fgr_tp2)


In [ ]:
group_cols <- c("Control"  = "#708090",
                "Case-SGA" = "#4A9ED9",
                "Case-FGR" = "#D88999")

In [ ]:
plot_pcoa <- function(dist, meta, adjust_res, title) {
  r2 <-  adjust_res["R2"]
  p  <-  adjust_res["p"]

  pc <- cmdscale(dist, k = 2, eig = TRUE)
  var_exp <- round(100 * pc$eig[1:2] / sum(pc$eig[pc$eig > 0]), 1)

  df <- data.frame(PC1 = pc$points[, 1], PC2 = pc$points[, 2], Group = meta$Group)
  p_lab <- ifelse(p < 0.001, "p < 0.001", sprintf("p = %.3f", p))

  ggplot(df, aes(PC1, PC2, colour = Group)) +
    geom_point(size = 2, alpha = 0.7) +
    stat_ellipse(level = 0.95, linewidth = 0.6) +
    scale_colour_manual(values = group_cols) +
    labs(title = title,
         subtitle = sprintf("Adjusted R\u00b2 = %.3f, %s", r2, p_lab),
         x = paste0("PCoA1 (", var_exp[1], "%)"),
         y = paste0("PCoA2 (", var_exp[2], "%)")) +
    theme_bw() +
    theme(plot.title = element_text(face = "bold", size = 11),
          plot.subtitle = element_text(size = 9),
         axis.text  = element_text(colour = "black"),
         axis.title = element_text(face = "bold",colour = "black"),
         axis.ticks = element_line(colour = "black"),
         legend.title  = element_text(face = "bold",colour = "black"),
         legend.text  = element_text(colour = "black"))
}

In [ ]:
pA <- plot_pcoa(bc_SGA_tp1, sga_tp1, bc_p_sga_tp1, "SGA vs AGA, TP1")
pB <- plot_pcoa(bc_SGA_tp2, sga_tp2_final, bc_p_sga_tp2, "SGA vs AGA, TP2")
pC <- plot_pcoa(bc_FGR_tp1, fgr_tp1, bc_p_fgr_tp1, "FGR vs AGA, TP1")
pD <- plot_pcoa(bc_FGR_tp2, fgr_tp2_final, bc_p_fgr_tp2, "FGR vs AGA, TP2")

In [ ]:
fig_S2 <- (pA | pB) / (pC | pD) +
  plot_layout(guides = "collect") +
  plot_annotation(tag_levels = "A",tag_suffix = ".")
fig_S2 

In [ ]:
ggsave("Figure_S2_PCoA_BrayCurtis.pdf", fig_S2,
       width = 10, height = 8, device = cairo_pdf)


In [ ]:
ggsave("Figure_S2_PCoA_BrayCurtis.eps", fig_S2,
       width = 10, height = 8, device = cairo_ps, fallback_resolution = 600)